# Work-Life Firewall Kaggle Runner

Run this notebook top-to-bottom in Kaggle with GPU enabled.

## Required Kaggle Secrets
- `WANDB_API_KEY`
- `HF_TOKEN` (optional, only needed for model upload)


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# Override via env vars if needed.
GITHUB_REPO = os.environ.get('GITHUB_REPO', 'https://github.com/EchoOfCode/meta_hackathon.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'main')
WORK_REPO_DIR = Path('/kaggle/working/meta_hackathon')
SAFE_CWD = Path('/kaggle/working')


def _run(cmd, cwd=None):
    run_cwd = str(cwd or SAFE_CWD)
    proc = subprocess.run(cmd, cwd=run_cwd, text=True, capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(cmd)}\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _repo_url_with_token(repo_url: str) -> str:
    token = os.environ.get('GITHUB_TOKEN')
    if token and repo_url.startswith('https://github.com/'):
        return repo_url.replace('https://', f'https://{token}@', 1)
    return repo_url


def sync_repo_from_github() -> Path:
    """Always get latest branch tip into /kaggle/working/meta_hackathon."""
    repo_url = _repo_url_with_token(GITHUB_REPO)

    # Ensure we are never inside a path that may be deleted during sync.
    SAFE_CWD.mkdir(parents=True, exist_ok=True)
    os.chdir(SAFE_CWD)

    if (WORK_REPO_DIR / '.git').exists():
        _run(['git', 'fetch', 'origin', GITHUB_BRANCH], cwd=WORK_REPO_DIR)
        _run(['git', 'reset', '--hard', f'origin/{GITHUB_BRANCH}'], cwd=WORK_REPO_DIR)
    else:
        if WORK_REPO_DIR.exists():
            shutil.rmtree(WORK_REPO_DIR)
        _run([
            'git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, repo_url, str(WORK_REPO_DIR)
        ], cwd=SAFE_CWD)

    return WORK_REPO_DIR


def find_repo_dir_from_inputs() -> Path:
    """Fallback for offline runs using attached Kaggle dataset input."""
    candidates = []

    working = Path('/kaggle/working')
    if working.exists():
        candidates.append(working)
        candidates.extend([p for p in working.iterdir() if p.is_dir()])

    input_root = Path('/kaggle/input')
    if input_root.exists():
        candidates.append(input_root)
        for ds in input_root.iterdir():
            if ds.is_dir():
                candidates.append(ds)
                candidates.extend([p for p in ds.iterdir() if p.is_dir()])

    for c in candidates:
        if (c / 'openenv.yaml').exists() and (c / 'training').exists():
            return c

    raise FileNotFoundError(
        'Could not find project root with openenv.yaml + training/. '
        'Attach dataset input or enable internet for GitHub sync.'
    )


# Default behavior: pull latest from GitHub each run.
USE_GITHUB_SYNC = os.environ.get('USE_GITHUB_SYNC', '1') == '1'

if USE_GITHUB_SYNC:
    try:
        REPO_DIR = sync_repo_from_github()
        print('Synced latest code from GitHub:', GITHUB_REPO, '@', GITHUB_BRANCH)
    except Exception as exc:
        print('GitHub sync failed, falling back to /kaggle/input snapshot:')
        print(exc)
        REPO_DIR = find_repo_dir_from_inputs()
else:
    REPO_DIR = find_repo_dir_from_inputs()

# /kaggle/input is read-only; copy to /kaggle/working for training outputs.
if str(REPO_DIR).startswith('/kaggle/input'):
    if WORK_REPO_DIR.exists() and WORK_REPO_DIR != REPO_DIR:
        shutil.rmtree(WORK_REPO_DIR)
    shutil.copytree(REPO_DIR, WORK_REPO_DIR, dirs_exist_ok=True)
    REPO_DIR = WORK_REPO_DIR

os.chdir(REPO_DIR)
print('CWD:', Path.cwd())
print('Using project root:', REPO_DIR)
!python --version
!nvidia-smi

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

# WandB token (required for tracking)
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
os.environ['WANDB_PROJECT'] = 'meta_hackathon'

# HF token is optional; needed only if you push model from Kaggle
try:
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['HUGGINGFACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token loaded from Kaggle Secrets')
except Exception:
    print('HF token not set (this is okay if you are not uploading model now)')

print('WandB key loaded:', bool(os.environ.get('WANDB_API_KEY')))

In [ ]:
# Fast stable run for Kaggle T4
!python training/train.py --mode real --speed-preset fast --size-preset small --steps 50 --run-name meta-kaggle-final --wandb-project meta_hackathon

In [ ]:
import json
from pathlib import Path

metrics_path = Path('evaluation/results/training_metrics.json')
metrics = json.loads(metrics_path.read_text())
print('Metrics file:', metrics_path)
print('Mode:', metrics.get('mode'))
print('Model:', metrics.get('model_id'))
print('Train runtime (s):', metrics.get('train_runtime_seconds'))
print('WandB URL:', metrics.get('wandb_run_url'))
print('Loss points:', len(metrics.get('loss_curve', [])))
print('Reward points:', len(metrics.get('reward_curve', [])))

In [ ]:
!python evaluation/evaluate.py --episodes 50
!python evaluation/plot_results.py
!ls -lah evaluation/results

In [ ]:
# Package submission artifacts for download
!zip -r submission_artifacts.zip README.md openenv.yaml environment training evaluation/results -x "*/__pycache__/*" "*.pyc" ".ipynb_checkpoints/*"
!ls -lah submission_artifacts.zip

In [ ]:
# Optional: upload model to HF Model Hub
# Uncomment and edit repo id when needed
# !python training/push_to_hf.py --repo-id YUS200619/meta_hackathon-qwen-model --folder checkpoints/final